# Formal D10 Velocity Detail Replot

This notebook follows the old `heart_d10_velocity.ipynb` structure for
multi-setting D10 stream visualization and boundary zoom inspection,
while using the corrected formal chicken-heart results and the formal/native
velocity rendering logic as the source of truth.

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import anndata as ad
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import seaborn as sns
import torch

import os
REPO_ROOT = Path(os.environ.get("CYTOBRIDGE_SOURCE_DIR", ".")).resolve()
PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data" / "chicken_heart"
sys.path.insert(0, str(REPO_ROOT / "reproduction" / "chicken_heart"))
ANALYSIS_ROOT = Path(os.environ.get("CYTOBRIDGE_HEART_OUTPUT_DIR", PROJECT_DIR / "outputs" / "chicken_heart_paper")).resolve()
import CytoBridge as cb
PACKAGE_ROOT = Path(cb.__file__).resolve().parents[1]

for path in (REPO_ROOT, PACKAGE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import CytoBridge as cb
from downstream_helpers.heart import HEART_LABEL_TO_COLOR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
scv.settings.set_figure_params("scvelo")
sns.set_theme(style="white")
plt.rcParams["font.family"] = "DejaVu Sans"
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

INTERP_RUN_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_daily_piecewise_interpolation_celltypecorrected"
SLICE_DIR = INTERP_RUN_DIR / "slice_data"
MANIFEST_PATH = INTERP_RUN_DIR / "manifest.json"
ALIGNED_H5AD_PATH = DATA_DIR / "aligned.h5ad"
MODEL_DIR = DATA_DIR / "model"
EDGE_PREDICTOR_PATH = DATA_DIR / "edge_classifier" / "chicken_heart_edge_model.pt"
OUTPUT_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_d10_velocity_detail_celltypecorrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_LABEL = "D10"
COMPONENT_KEY = "full"
MODE = "gene"

LEGACY_REGION_COLOR_MAP = {
    "Ventricle": "#7f7f7f",
    "Outflow tract": "#66c2a5",
    "Endothelium": "#9edae5",
    "Valves": "#c25bac",
    "Atria": "#cab2d6",
    "Epicardium": "#dbdb8d",
    "Trabecular LV and endocardium": "#4575b4",
    "Compact LV and inter-ventricular septum": "#aec7e8",
    "Right ventricle": "#c49c94",
}

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print(f"Using device: {DEVICE}")
print(f"Notebook output dir: {OUTPUT_DIR}")
print(f"Interpolation run dir: {INTERP_RUN_DIR}")

In [ ]:
def canonical_label(value: object) -> str:
    return str(value).replace("\n", " ").replace("\r", " ").replace("  ", " ").strip()


def build_region_palette(values: list[str]) -> dict[str, str]:
    regions = sorted({canonical_label(v) for v in values})
    base = {canonical_label(label): str(color) for label, color in LEGACY_REGION_COLOR_MAP.items()}
    missing = [region for region in regions if region not in base]
    if missing:
        extra = sns.color_palette("husl", len(missing)).as_hex()
        for region, color in zip(missing, extra):
            base[region] = mcolors.to_hex(color)
    return {region: base[region] for region in regions}


def locate_slice_entry(time_label: str) -> dict:
    for item in manifest["slices"]:
        if str(item["time_label"]) == str(time_label):
            return item
    raise KeyError(f"Could not find slice entry for {time_label!r}")


def attach_region_for_observed_slice(adata_t: ad.AnnData, reference_h5ad_path: Path) -> ad.AnnData:
    if "region" in adata_t.obs.columns:
        adata_t.obs["region"] = adata_t.obs["region"].astype(str).map(canonical_label).values
        return adata_t

    reference = ad.read_h5ad(reference_h5ad_path)
    region_col = None
    for candidate in ("region", "Region", "anatomic_region"):
        if candidate in reference.obs.columns:
            region_col = candidate
            break
    if region_col is None:
        raise KeyError(f"Reference aligned h5ad is missing region columns. Found: {list(reference.obs.columns)}")

    lookup = reference.obs[region_col].astype(str).map(canonical_label)
    shared = adata_t.obs_names.intersection(lookup.index)
    if len(shared) == 0:
        raise KeyError(f"No shared obs_names found to attach region for {TIME_LABEL}.")

    region_series = pd.Series(index=adata_t.obs_names, dtype=object)
    region_series.loc[shared] = lookup.loc[shared].values
    adata_t.obs["region"] = region_series.fillna("Unknown").astype(str).map(canonical_label).values
    return adata_t


slice_entry = locate_slice_entry(TIME_LABEL)
slice_path = Path(slice_entry["slice_h5ad"])
if not slice_path.is_absolute():
    slice_path = SLICE_DIR / slice_path.name

d10_adata = ad.read_h5ad(slice_path)
d10_adata.obs["celltype_prediction"] = d10_adata.obs["celltype_prediction"].astype(str).map(canonical_label).values
d10_adata.obs["time_label"] = TIME_LABEL
d10_adata.obs["time_float"] = float(slice_entry["time_float"])
d10_adata.uns["time_label"] = TIME_LABEL
d10_adata.uns["time_float"] = float(slice_entry["time_float"])
d10_adata = attach_region_for_observed_slice(d10_adata, ALIGNED_H5AD_PATH)

if "spatial" not in d10_adata.obsm:
    if "spatial_aligned" in d10_adata.obsm:
        d10_adata.obsm["spatial"] = np.asarray(d10_adata.obsm["spatial_aligned"], dtype=np.float32)
    else:
        d10_adata.obsm["spatial"] = np.asarray(d10_adata.X[:, :2], dtype=np.float32)

region_to_color = build_region_palette(d10_adata.obs["region"].astype(str).tolist())
celltype_to_color = {
    canonical_label(label): color
    for label, color in HEART_LABEL_TO_COLOR.items()
}

print(f"D10 cells: {d10_adata.n_obs}")
print(f"Model time: {slice_entry['time_float']}")
print("Regions:", sorted(region_to_color))

In [ ]:
PARAMETER_SETTINGS = [
    {
        "name": "setting_1_baseline",
        "size": 220,
        "density": 5,
        "alpha": 0.80,
        "linewidth": 0.45,
        "arrowsize": 0.60,
        "min_mass": 0.010,
        "smooth": 0.20,
        "neighbor_k": 30,
        "max_length": 4.0,
        "integration_direction": "both",
    },
    {
        "name": "setting_2_smoother_stream",
        "size": 220,
        "density": 6,
        "alpha": 0.82,
        "linewidth": 0.50,
        "arrowsize": 0.65,
        "min_mass": 0.010,
        "smooth": 0.30,
        "neighbor_k": 36,
        "max_length": 4.0,
        "integration_direction": "both",
    },
    {
        "name": "setting_3_sparse_clean",
        "size": 220,
        "density": 4,
        "alpha": 0.78,
        "linewidth": 0.40,
        "arrowsize": 0.55,
        "min_mass": 0.015,
        "smooth": 0.16,
        "neighbor_k": 24,
        "max_length": 3.6,
        "integration_direction": "both",
    },
    {
        "name": "setting_4_dense_emphasis",
        "size": 220,
        "density": 7,
        "alpha": 0.84,
        "linewidth": 0.52,
        "arrowsize": 0.70,
        "min_mass": 0.008,
        "smooth": 0.26,
        "neighbor_k": 40,
        "max_length": 4.2,
        "integration_direction": "both",
    },
    {
        "name": "setting_5_boundary_low_smooth",
        "size": 220,
        "density": 7,
        "alpha": 0.84,
        "linewidth": 0.42,
        "arrowsize": 0.58,
        "min_mass": 0.009,
        "smooth": 0.12,
        "neighbor_k": 24,
        "max_length": 3.8,
        "integration_direction": "both",
    },
    {
        "name": "setting_6_boundary_micro_detail",
        "size": 220,
        "density": 8,
        "alpha": 0.86,
        "linewidth": 0.38,
        "arrowsize": 0.52,
        "min_mass": 0.007,
        "smooth": 0.08,
        "neighbor_k": 20,
        "max_length": 3.2,
        "integration_direction": "both",
    },
    {
        "name": "setting_7_high_contrast_boundary",
        "size": 220,
        "density": 8,
        "alpha": 0.88,
        "linewidth": 0.40,
        "arrowsize": 0.56,
        "min_mass": 0.007,
        "smooth": 0.10,
        "neighbor_k": 22,
        "max_length": 3.4,
        "integration_direction": "forward",
    },
    {
        "name": "setting_8_boundary_visible_dense",
        "size": 220,
        "density": 9,
        "alpha": 0.86,
        "linewidth": 0.44,
        "arrowsize": 0.60,
        "min_mass": 0.009,
        "smooth": 0.11,
        "neighbor_k": 26,
        "max_length": 3.8,
        "integration_direction": "both",
    },
    {
        "name": "ct_zoom_1_visible_local",
        "size": 220,
        "density": 9,
        "alpha": 0.92,
        "linewidth": 0.50,
        "arrowsize": 0.62,
        "min_mass": 0.010,
        "smooth": 0.14,
        "neighbor_k": 22,
        "max_length": 2.2,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_2_dense_local",
        "size": 220,
        "density": 10,
        "alpha": 0.92,
        "linewidth": 0.48,
        "arrowsize": 0.56,
        "min_mass": 0.008,
        "smooth": 0.12,
        "neighbor_k": 20,
        "max_length": 2.1,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_3_detail_balance",
        "size": 210,
        "density": 9,
        "alpha": 0.90,
        "linewidth": 0.46,
        "arrowsize": 0.56,
        "min_mass": 0.009,
        "smooth": 0.13,
        "neighbor_k": 24,
        "max_length": 2.0,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_4_short_visible",
        "size": 215,
        "density": 9,
        "alpha": 0.91,
        "linewidth": 0.48,
        "arrowsize": 0.60,
        "min_mass": 0.010,
        "smooth": 0.15,
        "neighbor_k": 18,
        "max_length": 1.9,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_5_gene_dense_more",
        "size": 220,
        "density": 10,
        "alpha": 0.92,
        "linewidth": 0.47,
        "arrowsize": 0.58,
        "min_mass": 0.008,
        "smooth": 0.11,
        "neighbor_k": 26,
        "max_length": 2.2,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_6_gene_soft_visible",
        "size": 220,
        "density": 9,
        "alpha": 0.92,
        "linewidth": 0.49,
        "arrowsize": 0.60,
        "min_mass": 0.009,
        "smooth": 0.16,
        "neighbor_k": 20,
        "max_length": 2.2,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_6b_gene_soft_visible_epicardium",
        "size": 205,
        "density": 10,
        "alpha": 0.92,
        "linewidth": 0.48,
        "arrowsize": 0.60,
        "min_mass": 0.007,
        "smooth": 0.18,
        "neighbor_k": 22,
        "max_length": 2.5,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_7_gene_balance_24",
        "size": 215,
        "density": 9,
        "alpha": 0.91,
        "linewidth": 0.47,
        "arrowsize": 0.58,
        "min_mass": 0.010,
        "smooth": 0.12,
        "neighbor_k": 24,
        "max_length": 2.0,
        "integration_direction": "both",
        "graph_rep": "X",
    },
    {
        "name": "ct_zoom_8_spatial_local_compare",
        "size": 200,
        "density": 9,
        "alpha": 0.90,
        "linewidth": 0.46,
        "arrowsize": 0.56,
        "min_mass": 0.010,
        "smooth": 0.14,
        "neighbor_k": 8,
        "max_length": 2.0,
        "integration_direction": "both",
        "graph_rep": "X_spatial",
    },
]

(OUTPUT_DIR / "parameter_settings.json").write_text(
    json.dumps(PARAMETER_SETTINGS, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
PARAMETER_SETTINGS[:2]

In [ ]:
loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=int(d10_adata.X.shape[1]),
    device=DEVICE,
    edge_predictor_path=EDGE_PREDICTOR_PATH,
)
model = loaded.model
interaction_threshold = float(getattr(getattr(model, "interaction_net", None), "cutoff", 1000.0))

d10_velocity_components = cb.tl.compute_velocity_components(
    data=np.asarray(d10_adata.X, dtype=np.float32),
    time_value=float(slice_entry["time_float"]),
    model=model,
    interaction_m=1024,
    interaction_threshold=interaction_threshold,
    device=DEVICE,
    spatial_dim=2,
)

print(f"Loaded formal weight stage: {loaded.weight_stage}")
print(f"Loaded formal score stage: {loaded.score_stage}")
print(f"Interaction cutoff: {interaction_threshold}")
print("Velocity component keys:", sorted(k for k in d10_velocity_components.keys() if isinstance(d10_velocity_components[k], np.ndarray)))

In [ ]:
def compute_canvas_limits(adata_obj: ad.AnnData):
    coords = np.asarray(adata_obj.obsm["X_spatial"], dtype=float)
    x_min, y_min = coords.min(axis=0)
    x_max, y_max = coords.max(axis=0)
    pad_x = (x_max - x_min) * 0.05
    pad_y = (y_max - y_min) * 0.05
    width = max(x_max - x_min, 1e-8)
    height = max(y_max - y_min, 1e-8)
    return [x_min - pad_x, x_max + pad_x], [y_min - pad_y, y_max + pad_y], height / width


def compute_region_boundary_box(adata_obj: ad.AnnData, region_a: str, region_b: str, pad_ratio: float = 0.12):
    coords = np.asarray(adata_obj.obsm["X_spatial"], dtype=float)
    regions = adata_obj.obs["region"].astype(str).to_numpy()
    mask = np.isin(regions, [region_a, region_b])
    if not np.any(mask):
        raise ValueError(f"No cells found for boundary regions: {region_a!r}, {region_b!r}")
    sub = coords[mask]
    x_min, y_min = sub.min(axis=0)
    x_max, y_max = sub.max(axis=0)
    span_x = max(x_max - x_min, 1e-8)
    span_y = max(y_max - y_min, 1e-8)
    pad_x = span_x * pad_ratio
    pad_y = span_y * pad_ratio
    return [x_min - pad_x, x_max + pad_x], [y_min - pad_y, y_max + pad_y]


def compute_focus_box(xlim, ylim, *, zoom_ratio: float = 0.65, center_shift=(0.0, 0.0)):
    x_min, x_max = xlim
    y_min, y_max = ylim
    width = max(x_max - x_min, 1e-8)
    height = max(y_max - y_min, 1e-8)
    cx = (x_min + x_max) / 2 + center_shift[0] * width
    cy = (y_min + y_max) / 2 + center_shift[1] * height
    half_w = width * zoom_ratio / 2
    half_h = height * zoom_ratio / 2
    return [cx - half_w, cx + half_w], [cy - half_h, cy + half_h]


velocity_cache: dict[tuple[str, int], ad.AnnData] = {}


def build_formal_d10_velocity_adata(
    *,
    component_key: str,
    neighbor_k: int,
    graph_rep: str = "X",
    color_key: str = "region",
    palette: dict[str, str] | None = None,
) -> ad.AnnData:
    cache_key = (component_key, int(neighbor_k), str(graph_rep))
    if cache_key in velocity_cache:
        return velocity_cache[cache_key].copy()

    coords = np.asarray(d10_adata.obsm["spatial"], dtype=np.float32)
    gene_data = np.asarray(d10_adata.X[:, 2:], dtype=np.float32)
    gene_velocity = np.asarray(d10_velocity_components[component_key][:, 2:], dtype=np.float32)

    ad_plot = ad.AnnData(X=gene_data)
    ad_plot.obsm["X_spatial"] = coords.copy()
    ad_plot.obsm["spatial"] = coords.copy()
    ad_plot.layers["Ms"] = gene_data.copy()
    ad_plot.layers["velocity"] = gene_velocity.copy()
    ad_plot.obs[color_key] = d10_adata.obs[color_key].astype(str).map(canonical_label).values
    ad_plot.obs[color_key] = ad_plot.obs[color_key].astype("category")
    if palette is not None:
        cats = list(ad_plot.obs[color_key].cat.categories)
        ad_plot.uns[f"{color_key}_colors"] = [palette.get(cat, "#888888") for cat in cats]

    # For detail inspection we allow either the formal-native graph in gene
    # state space ("X") or the old notebook's more local spatial graph
    # ("X_spatial"), while always keeping the formal velocity components.
    sc.pp.neighbors(ad_plot, n_neighbors=int(neighbor_k), use_rep=str(graph_rep))
    scv.tl.velocity_graph(ad_plot, vkey="velocity", xkey="Ms", n_jobs=1)
    scv.tl.velocity_embedding(ad_plot, basis="spatial", vkey="velocity")

    velocity_cache[cache_key] = ad_plot.copy()
    return ad_plot


def render_velocity_stream(
    adata_obj: ad.AnnData,
    *,
    time_label: str,
    component_key: str,
    mode: str,
    params: dict,
    save_dir: Path,
    xlim=None,
    ylim=None,
    title_suffix: str = "",
):
    adata_plot = adata_obj.copy()
    default_xlim, default_ylim, aspect_ratio = compute_canvas_limits(adata_plot)
    if xlim is None:
        xlim = default_xlim
    if ylim is None:
        ylim = default_ylim

    fig, ax = plt.subplots(figsize=(6, 6 * aspect_ratio * 1.1))
    scv.pl.velocity_embedding_stream(
        adata_plot,
        basis="spatial",
        color="region",
        ax=ax,
        show=False,
        legend_loc="none",
        size=params["size"],
        density=params["density"],
        alpha=params["alpha"],
        linewidth=params["linewidth"],
        arrowsize=params["arrowsize"],
        min_mass=params["min_mass"],
        smooth=params["smooth"],
        n_neighbors=params.get("neighbor_k"),
        max_length=params.get("max_length", 4.0),
        integration_direction=params.get("integration_direction", "both"),
        title=f"{time_label} | {component_key} | {params['name']}{title_suffix}",
    )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")

    stem = f"velocity_{time_label}_{component_key}_{mode}_{params['name']}"
    png_path = save_dir / f"{stem}.png"
    svg_path = save_dir / f"{stem}.svg"
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.show()
    return {"name": params["name"], "png": str(png_path), "svg": str(svg_path)}

In [ ]:
global_outputs = []
for params in PARAMETER_SETTINGS[:8]:
    ad_plot = build_formal_d10_velocity_adata(
        component_key=COMPONENT_KEY,
        neighbor_k=int(params["neighbor_k"]),
        graph_rep=str(params.get("graph_rep", "X")),
        color_key="region",
        palette=region_to_color,
    )
    global_outputs.append(
        render_velocity_stream(
            ad_plot,
            time_label=TIME_LABEL,
            component_key=COMPONENT_KEY,
            mode=MODE,
            params=params,
            save_dir=OUTPUT_DIR,
        )
    )

global_outputs

In [ ]:
boundary_name = "compactlv_vs_trabecularlv"
boundary_regions = (
    "Compact LV and inter-ventricular septum",
    "Trabecular LV and endocardium",
)

zoom_outputs = []
boundary_base_params = PARAMETER_SETTINGS[8:]
base_plot = build_formal_d10_velocity_adata(
    component_key=COMPONENT_KEY,
    neighbor_k=int(boundary_base_params[0]["neighbor_k"]),
    graph_rep=str(boundary_base_params[0].get("graph_rep", "X")),
    color_key="region",
    palette=region_to_color,
)
boundary_xlim, boundary_ylim = compute_region_boundary_box(
    base_plot,
    boundary_regions[0],
    boundary_regions[1],
    pad_ratio=0.12,
)
epicardium_focus_xlim, epicardium_focus_ylim = compute_focus_box(
    boundary_xlim,
    boundary_ylim,
    zoom_ratio=0.42,
    center_shift=(-0.22, 0.22),
)

for params in boundary_base_params:
    local_plot = build_formal_d10_velocity_adata(
        component_key=COMPONENT_KEY,
        neighbor_k=int(params["neighbor_k"]),
        graph_rep=str(params.get("graph_rep", "X")),
        color_key="region",
        palette=region_to_color,
    )
    stemmed = dict(params)
    stemmed["name"] = f"{boundary_name}_{params['name']}_zoom"
    zoom_outputs.append(
        render_velocity_stream(
            local_plot,
            time_label=TIME_LABEL,
            component_key=COMPONENT_KEY,
            mode=MODE,
            params=stemmed,
            save_dir=OUTPUT_DIR / "boundary_zoom",
            xlim=boundary_xlim,
            ylim=boundary_ylim,
            title_suffix=f" | {boundary_regions[0]} vs {boundary_regions[1]}",
        )
    )

focus_param_names = {
    "ct_zoom_6_gene_soft_visible",
    "ct_zoom_6b_gene_soft_visible_epicardium",
}
epicardium_focus_outputs = []
for params in boundary_base_params:
    if params["name"] not in focus_param_names:
        continue
    local_plot = build_formal_d10_velocity_adata(
        component_key=COMPONENT_KEY,
        neighbor_k=int(params["neighbor_k"]),
        graph_rep=str(params.get("graph_rep", "X")),
        color_key="region",
        palette=region_to_color,
    )
    stemmed = dict(params)
    stemmed["name"] = f"{boundary_name}_{params['name']}_epicardium_focus"
    epicardium_focus_outputs.append(
        render_velocity_stream(
            local_plot,
            time_label=TIME_LABEL,
            component_key=COMPONENT_KEY,
            mode=MODE,
            params=stemmed,
            save_dir=OUTPUT_DIR / "epicardium_focus",
            xlim=epicardium_focus_xlim,
            ylim=epicardium_focus_ylim,
            title_suffix=(
                f" | left-upper epicardium focus"
                f" | {boundary_regions[0]} vs {boundary_regions[1]}"
            ),
        )
    )

{
    "boundary_zoom_outputs": zoom_outputs,
    "epicardium_focus_outputs": epicardium_focus_outputs,
    "epicardium_focus_xlim": epicardium_focus_xlim,
    "epicardium_focus_ylim": epicardium_focus_ylim,
}

In [ ]:
summary = {
    "time_label": TIME_LABEL,
    "component_key": COMPONENT_KEY,
    "mode": MODE,
    "source_slice_h5ad": str(slice_path),
    "model_dir": str(MODEL_DIR),
    "interaction_cutoff": interaction_threshold,
    "palette_region": region_to_color,
    "parameter_settings_path": str(OUTPUT_DIR / "parameter_settings.json"),
    "global_outputs": global_outputs,
    "boundary_zoom_outputs": zoom_outputs,
    "epicardium_focus_outputs": epicardium_focus_outputs,
}

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved summary to {summary_path}")
print("Global outputs:")
for item in global_outputs:
    print(f"- {item['name']} -> {item['svg']}")
print("Boundary zoom outputs:")
for item in zoom_outputs:
    print(f"- {item['name']} -> {item['svg']}")
print("Epicardium focus outputs:")
for item in epicardium_focus_outputs:
    print(f"- {item['name']} -> {item['svg']}")